In [6]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../../").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(
        str(PROJECT_ROOT)
    )

print(PROJECT_ROOT)

/home/ubuntu/Projects/thesis-code


In [7]:
from src.baseline.dataset import COdeBaselineDataset
from src.baseline.transforms import get_image_transform
from src.baseline import config


dataset = COdeBaselineDataset(
    csv_path=config.DATASET_PATH,
    split="train",
    image_root=config.IMAGE_ROOT,
    transform=get_image_transform(),
)


sample = dataset[0]


print(sample["checkup_id"])
print(len(sample["images"]))
print(sample["images"][0].shape)
print(sample["labels"].shape)

train: 6129 samples
0001-001
1
torch.Size([3, 224, 224])
torch.Size([13])


# Thesis Note 05.2

# Baseline Experiment B — Six-Label Radiograph-only Fine-tuned Classification

## Overview

This experiment represents the official radiograph-only baseline for the six-label COde classification benchmark.

The objective is to establish a clean and reproducible unimodal baseline using radiographic images alone before introducing photographs, clinical text, and multimodal fusion.

The six-label benchmark was constructed from the previously finalized 13-label patient-level dataset while preserving the original patient identities, visits, modalities, and leakage-safe split assignments.

The experiment evaluates whether radiographic images alone contain useful information for automated multi-label dental diagnosis classification.

---

# 1. Experiment Objective

The goal of this experiment is to train and evaluate a radiograph-only deep learning classifier for six-label dental diagnosis prediction.

The model receives only radiographic images from each patient visit and predicts the following six diagnostic categories:

1. Caries
2. Gingivitis
3. Malocclusion
4. Pulpitis
5. Tooth Loss
6. Tooth Structure Loss

The experiment answers the following research question:

> How effective are radiographic images alone for automated multi-label dental diagnosis classification on the six-label COde benchmark?

---

# 2. Six-Label Benchmark Construction

The six-label benchmark was derived from the finalized 13-label patient-level dataset.

The original 13-label representation was converted into six benchmark labels.

The malocclusion label combines the three original malocclusion categories:

- Class I Malocclusion
- Class II Malocclusion
- Class III Malocclusion

The remaining selected conditions are retained as individual labels.

The final label set is:

- label_caries
- label_gingivitis
- label_malocclusion
- label_pulpitis
- label_tooth_loss
- label_tooth_structure_loss

The resulting dataset was validated successfully.

Final dataset:

results/six_label_patient_level_dataset/labeled_dataset.csv

Label distribution:

results/six_label_patient_level_dataset/label_split_distribution.csv

Dataset summary:

results/six_label_patient_level_dataset/dataset_summary.json

---

# 3. Dataset Configuration

Dataset:

COde Dataset

Task:

Six-label multi-label dental diagnosis classification

Number of labels:

6

Total visits:

8,775

Total patients:

4,800

The six-label dataset preserves the previously established patient-level split.

---

# 4. Patient-Level Split

The experiment uses the previously validated patient-level split.

Split strategy:

Patient-level split

Seed:

42

Train      : 3,360 patients / 6,129 visits
Validation :   720 patients / 1,330 visits
Test       :   720 patients / 1,316 visits

The patient-level split remains authoritative and was not modified during six-label benchmark construction.

Previous leakage auditing confirmed that the split is leakage-safe.

The six-label construction changes only the label representation and does not introduce a new patient split.

---

# 5. Label Coverage

The six-label benchmark contains fewer labeled visits than the original 13-label benchmark because a visit is considered labeled only when at least one of the six selected labels is positive.

Split coverage:

Train      : 4,649 / 6,129 labeled visits — 75.85%
Validation :   988 / 1,330 labeled visits — 74.29%
Test       : 1,004 / 1,316 labeled visits — 76.29%

This reduction in coverage is expected because the six-label benchmark intentionally focuses on a subset of the reconstructed diagnostic categories.

---

# 6. Input Modality

This experiment uses only radiographic images.

Used modality:

Radiographs

Excluded modalities:

Photographs

Clinical text

The input pipeline is:

Radiographs
    |
    v
ResNet50 Encoder
    |
    v
Mean Feature Aggregation
    |
    v
Classification Head
    |
    v
Six-label Prediction

Only visits containing at least one radiograph are included in the radiograph-only experiment.

---

# 7. Model Architecture

## Image Encoder

Backbone:

ResNet50

Initialization:

ImageNet pretrained weights

Training strategy:

Full encoder fine-tuning

The encoder parameters are updated during training rather than being frozen.

---

# 8. Variable-Length Radiograph Aggregation

Each dental visit may contain a variable number of radiographs.

Each radiograph is independently processed by the ResNet50 encoder and the resulting feature vectors are aggregated using mean pooling.

Architecture:

Radiograph 1 ----\
Radiograph 2 ----- ResNet50 ---- Mean Pooling ---- Classifier
Radiograph N ----/

Mathematically:

z = (1/N) Σ f(x_i)

where:

- x_i represents the i-th radiograph
- f(.) represents the ResNet50 encoder
- z represents the visit-level image representation

This produces a fixed-dimensional representation regardless of the number of radiographs associated with a visit.

---

# 9. Training Configuration

Modality:

Radiograph-only

Training samples:

2,972

Validation samples:

642

Batch size:

16

Number of epochs:

20

Loss function:

BCEWithLogitsLoss

Task formulation:

Multi-label classification

Learning rate:

1e-5

Weight decay:

1e-4

Encoder:

ResNet50

Pretrained:

ImageNet

Encoder:

Fine-tuned

Device:

CUDA

---

# 10. Training Results

The model showed substantially stronger learning behavior than the earlier 13-label radiograph-only experiment.

During the first epoch:

Train Loss:

0.4574

Validation Macro F1:

0.0913

Validation Micro F1:

0.4042

Validation AUROC:

0.5633

By epoch 12:

Validation Macro F1:

0.1503

Validation Micro F1:

0.5238

Validation AUROC:

0.7160

The best validation Macro F1 during training was approximately:

0.1830

at epoch 19.

The validation AUROC reached approximately:

0.7160

during training.

The training loss continuously decreased from:

0.4574

to:

0.1189

by epoch 20.

However, validation loss and threshold-dependent metrics fluctuated, indicating that the model began to overfit the training data during later epochs.

---

# 11. Threshold Optimization

Because this is a multi-label classification problem with substantially different label prevalences, a fixed probability threshold of 0.5 is not necessarily optimal for every diagnosis.

Therefore, label-specific thresholds were optimized on the validation split.

The procedure was:

1. Train the model using the training split.
2. Generate predictions on the validation split.
3. Search for the threshold maximizing F1-score independently for each label.
4. Save the resulting thresholds.
5. Use the validation-derived thresholds for subsequent test evaluation.

The optimized thresholds were:

label_caries                 : 0.45
label_gingivitis             : 0.25
label_malocclusion           : 0.35
label_pulpitis               : 0.35
label_tooth_loss             : 0.25
label_tooth_structure_loss   : 0.35

Thresholds were learned exclusively from the validation split.

The test split was not used for threshold selection.

Saved thresholds:

results/baseline/radiograph_only_6label/thresholds.json

---

# 12. Validation Results

Evaluation split:

Validation

Number of samples:

642

## Default Threshold (0.5)

| Metric | Score |
|---|---:|
| Macro F1 | 0.1830 |
| Micro F1 | 0.4736 |
| Accuracy | 0.3894 |
| AUROC | 0.7003 |

## Optimized Label-specific Thresholds

| Metric | Score |
|---|---:|
| Macro F1 | 0.3097 |
| Micro F1 | 0.4716 |
| Accuracy | 0.2399 |
| AUROC | 0.7003 |

Threshold optimization substantially improved Macro F1:

0.1830 → 0.3097

while AUROC remained unchanged because AUROC is threshold-independent.

---

# 13. Test Results

Evaluation split:

Test

Number of samples:

642

Thresholds:

Learned exclusively from the validation split

## Default Threshold (0.5)

| Metric | Score |
|---|---:|
| Macro F1 | 0.1722 |
| Micro F1 | 0.4734 |
| Accuracy | 0.3801 |
| AUROC | 0.6988 |

## Optimized Validation-derived Thresholds

| Metric | Score |
|---|---:|
| Macro F1 | 0.3048 |
| Micro F1 | 0.4498 |
| Accuracy | 0.2399 |
| AUROC | 0.6988 |

Threshold optimization improved test Macro F1:

0.1722 → 0.3048

The optimized thresholds were not re-estimated on the test set.

Therefore, the optimized test result represents a leakage-safe evaluation using thresholds learned from validation data.

---

# 14. Generalization Analysis

The validation and test AUROC values are very close:

Validation AUROC:

0.7003

Test AUROC:

0.6988

Difference:

approximately 0.0015

This indicates very stable ranking performance between validation and test data.

Similarly, optimized Macro F1 is highly consistent:

Validation:

0.3097

Test:

0.3048

Difference:

approximately 0.0049

This close agreement suggests that the radiograph-only model generalizes reasonably well under the patient-level split.

---

# 15. Effect of Threshold Optimization

Threshold optimization had a substantial effect on Macro F1.

Validation:

0.1830 → 0.3097

Test:

0.1722 → 0.3048

The improvement is expected because the six diagnostic categories have different prevalence levels.

A single threshold of 0.5 is therefore suboptimal for the imbalanced multi-label setting.

However, threshold optimization reduced exact multi-label accuracy:

Validation:

0.3894 → 0.2399

Test:

0.3801 → 0.2399

This demonstrates that threshold selection should be interpreted according to the evaluation metric.

For this thesis, Macro F1 is particularly informative because it gives equal importance to all six diagnostic categories despite differences in prevalence.

---

# 16. Observations

## Radiographs Contain Useful Diagnostic Signal

The model achieved an AUROC of approximately 0.70 on both validation and test sets.

This indicates that radiographic images contain meaningful information for predicting the selected dental diagnoses.

---

## Stronger Behavior Than the Earlier 13-Label Baseline

Compared with the previous 13-label radiograph-only experiment, the six-label formulation produces substantially more stable and interpretable learning behavior.

The model begins learning useful classification signals from the early epochs instead of producing near-zero Macro F1 during the initial training phase.

This supports the decision to use the six-label benchmark as the main baseline classification setting.

---

## Macro F1 vs Micro F1

The difference between Macro F1 and Micro F1 is substantial.

For example, on the test set with the default threshold:

Macro F1:

0.1722

Micro F1:

0.4734

This indicates that performance is not evenly distributed across the six diagnostic categories.

The high Micro F1 is influenced more strongly by the more frequent labels, whereas Macro F1 exposes weaker performance on less frequent diagnoses.

Therefore, Macro F1 should remain an important primary metric for comparison across future baseline and multimodal experiments.

---

# 17. Limitations

This baseline has several limitations:

- Radiographs are naturally missing for a substantial proportion of visits in the COde dataset.
- Only visits containing radiographs can be evaluated.
- Mean pooling does not explicitly model spatial relationships or differences between radiograph types.
- The label reconstruction process is based on clinical information and may contain weak-label noise.
- The model uses only radiographic information and therefore cannot exploit complementary visual or textual information.
- The dataset is multi-label and imbalanced, making threshold-dependent metrics sensitive to threshold selection.

---

# 18. Role in Thesis

This experiment establishes the official six-label radiograph-only baseline for the COde dataset.

It provides a reference point for evaluating the contribution of other modalities and multimodal learning strategies.

The planned progression is:

1. Photograph-only baseline
2. Radiograph-only baseline
3. Text-only baseline
4. Image + Radiograph baseline
5. Image + Text baseline
6. Full multimodal baseline
7. Missing-modality / robust multimodal experiments

The main purpose of this experiment is not to achieve the final best performance.

Instead, it establishes how much diagnostic information can be extracted from radiographs alone and provides a reproducible baseline against which future multimodal and missing-modality approaches can be compared.

---

# 19. Final Baseline Result

The final official test performance of the six-label radiograph-only fine-tuned baseline is:

| Metric | Default Threshold | Optimized Thresholds |
|---|---:|---:|
| Macro F1 | 0.1722 | 0.3048 |
| Micro F1 | 0.4734 | 0.4498 |
| Accuracy | 0.3801 | 0.2399 |
| AUROC | 0.6988 | 0.6988 |

The optimized-threshold Macro F1 of:

0.3048

and AUROC of:

0.6988

are the key results to carry forward when comparing this baseline against subsequent experiments.

---

# 20. Reproducibility Artifacts

Six-label dataset:

results/six_label_patient_level_dataset/labeled_dataset.csv

Label distribution:

results/six_label_patient_level_dataset/label_split_distribution.csv

Dataset summary:

results/six_label_patient_level_dataset/dataset_summary.json

Model checkpoint:

results/baseline/radiograph_only_6label/best_model.pt

Optimized thresholds:

results/baseline/radiograph_only_6label/thresholds.json

Validation evaluation:

results/baseline/radiograph_only_6label/validation_evaluation.json

Test evaluation:

results/baseline/radiograph_only_6label/test_evaluation.json

---

# Conclusion

The six-label radiograph-only baseline successfully learned meaningful diagnostic representations from radiographic images alone.

The model achieved:

Test Macro F1:

0.3048

Test Micro F1:

0.4498

Test AUROC:

0.6988

when using validation-derived label-specific thresholds.

The close agreement between validation and test performance indicates stable generalization under the patient-level split.

This baseline is now established as the radiographic reference point for the next stages of the COde multimodal classification experiments.

# Thesis Note 05.1
